# 06 — Baseline Modeling
Train 8 baseline classifiers on both Feature Set A (benchmark) and Feature Set B (realistic), compare them, and identify the best model per metric. Primary metric: **PR-AUC** (Average Precision).

In [ ]:
import sys, warnings, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (RAW_DIR, TARGET_COL, RANDOM_SEED,
                        FEATURE_SET_A_COLS, FEATURE_SET_B_COLS,
                        NUMERIC_COLS, NUMERIC_COLS_SET_B,
                        MODELS_DIR, REPORTS_DIR)
from src.data_loader import load_dataset
from src.features import encode_target, add_features, get_feature_lists
from src.preprocessing import build_preprocessing_pipeline, split_data
from src.modeling import get_baseline_models, train_model, save_model
from src.evaluation import (evaluate_binary_classifier, save_metrics_csv,
                             plot_roc_curve, plot_pr_curve, plot_confusion_matrix,
                             plot_model_comparison)
from src.utils import ensure_dir

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ensure_dir(MODELS_DIR)
ensure_dir(REPORTS_DIR)

# ── Load & prep ──────────────────────────────────────────────────────────────
df = load_dataset(RAW_DIR / "bank-additional-full.csv")
df = encode_target(df, TARGET_COL)
df = add_features(df)
X_train, X_test, y_train, y_test = split_data(df, TARGET_COL, random_state=RANDOM_SEED)
print(f"Train {X_train.shape}  |  Test {X_test.shape}  |  Positive rate {y_train.mean():.3f}")

## 1 — Train All Baselines (Feature Set B — No Duration)
We train on the **Realistic Business Model** first (Feature Set B). ⏳ This takes ~1–2 min.

In [ ]:
lists_b = get_feature_lists(df, TARGET_COL, exclude_duration=True)
preprocessor_b = build_preprocessing_pipeline(lists_b["numeric"], lists_b["categorical"])

models = get_baseline_models(random_state=RANDOM_SEED)
results_b = {}

for name, model in models.items():
    t0 = time.time()
    fitted_pipeline, cv_scores = train_model(model, X_train[lists_b["numeric"] + lists_b["categorical"]],
                                              y_train, preprocessor_b)
    elapsed = time.time() - t0
    metrics = evaluate_binary_classifier(fitted_pipeline,
                                          X_test[lists_b["numeric"] + lists_b["categorical"]], y_test)
    metrics.update({"model": name, "feature_set": "set_b",
                    "cv_pr_auc_mean": cv_scores["mean"], "cv_pr_auc_std": cv_scores["std"],
                    "train_time_s": round(elapsed, 2)})
    results_b[name] = {"pipeline": fitted_pipeline, "metrics": metrics}
    print(f"  {name:<35}  PR-AUC={metrics['average_precision']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}  ({elapsed:.1f}s)")

## 2 — Model Comparison Table (Feature Set B)

In [ ]:
metrics_rows_b = [v["metrics"] for v in results_b.values()]
df_metrics_b = pd.DataFrame(metrics_rows_b).set_index("model")

display_cols = ["average_precision", "roc_auc", "f1", "precision", "recall",
                "cv_pr_auc_mean", "cv_pr_auc_std", "train_time_s"]
display_cols = [c for c in display_cols if c in df_metrics_b.columns]

styled = (df_metrics_b[display_cols]
          .sort_values("average_precision", ascending=False)
          .style
          .highlight_max(subset=["average_precision", "roc_auc", "f1"], color="#d4edda")
          .format("{:.4f}", subset=[c for c in display_cols if c != "train_time_s"])
          .format("{:.2f}s", subset=["train_time_s"]))
display(styled)

## 3 — ROC & PR Curves for Top 3 Models (Set B)

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

top3 = df_metrics_b["average_precision"].sort_values(ascending=False).head(3).index.tolist()
X_test_b = X_test[lists_b["numeric"] + lists_b["categorical"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for color, name in zip(colors, top3):
    pipeline = results_b[name]["pipeline"]
    y_proba = pipeline.predict_proba(X_test_b)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={roc_auc_val:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc_val = auc(rec, prec)
    axes[1].plot(rec, prec, color=color, lw=2, label=f"{name} (AP={pr_auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves — Top 3 Models (Set B)"); axes[0].legend(fontsize=8)

baseline = y_test.mean()
axes[1].axhline(baseline, color="gray", linestyle="--", label=f"Baseline ({baseline:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR Curves — Top 3 Models (Set B)"); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4 — Confusion Matrices for Top 3 Models

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, top3):
    pipeline = results_b[name]["pipeline"]
    y_pred = pipeline.predict(X_test_b)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["No", "Yes"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=9)

plt.suptitle("Confusion Matrices — Top 3 Models (threshold=0.5)", fontsize=12)
plt.tight_layout()
plt.show()

## 5 — Save Metrics CSV & Best Model

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics_notebook.csv"
all_metrics = [v["metrics"] for v in results_b.values()]
pd.DataFrame(all_metrics).to_csv(metrics_path, index=False)
print(f"Metrics saved → {metrics_path}")

# Save best model
best_name = df_metrics_b["average_precision"].idxmax()
best_pipeline = results_b[best_name]["pipeline"]
best_path = MODELS_DIR / "best_model_set_b_notebook.joblib"
save_model(best_pipeline, best_path)
print(f"\n🏆 Best model (Set B): {best_name}")
print(f"   PR-AUC  = {df_metrics_b.loc[best_name, 'average_precision']:.4f}")
print(f"   ROC-AUC = {df_metrics_b.loc[best_name, 'roc_auc']:.4f}")
print(f"   Saved → {best_path}")